In [2]:
text = """
Maria Sharapova has basically no friends as tennis players on the WTA Tour. The Russian player has no problems in openly speaking about it and in a recent interview she said: I don't really hide any feelings too much.
I think everyone knows this is my job here. When I'm on the courts or when I'm on the court playing, I'm a competitor and I want to beat every single person whether they're in the locker room or across the net.
So I'm not the one to strike up a conversation about the weather and know that in the next few minutes I have to go and try to win a tennis match.
I'm a pretty competitive girl. I say my hellos, but I'm not sending any players flowers as well. Uhm, I'm not really friendly or close to many players.
I have not a lot of friends away from the courts.' When she said she is not really close to a lot of players, is that something strategic that she is doing? Is it different on the men's tour than the women's tour? No, not at all.
I think just because you're in the same sport doesn't mean that you have to be friends with everyone just because you're categorized, you're a tennis player, so you're going to get along with tennis players.
I think every person has different interests. I have friends that have completely different jobs and interests, and I've met them in very different parts of my life.
I think everyone just thinks because we're tennis players we should be the greatest of friends. But ultimately tennis is just a very small part of what we do.
There are so many other things that we're interested in, that we do.'
"""

In [3]:
text = """
Natural Language Processing (NLP) is a sub-field of artificial intelligence (AI) that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a valuable way. By utilizing NLP, developers can organize and structure knowledge to perform tasks such as automatic summarization, translation, named entity recognition, relationship extraction, sentiment analysis, speech recognition, and topic segmentation.
"""

In [10]:
!uv run python -m spacy download en_core_web_sm

Using Python 3.13.5 environment at: /Users/sai.rachakonda/my_files/notebooks/.venv
Resolved 1 package in 3.18s                                          
Prepared 1 package in 2.51s                                                  en-core-web-sm       ------------------------------ 12.20 MiB/12.21 MiB         
Installed 1 package in 3ms.0 (from https://github.com/explos
 + en-core-web-sm==3.8.0 (from https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [11]:
import spacy
from heapq import nlargest
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation
nlp = spacy.load("en_core_web_sm")

In [12]:
stopwords = set(STOP_WORDS)
punctuation = punctuation + '\n'
punctuation = set(punctuation) 
doc = nlp(text)

In [13]:
tokens = [token.text for token in doc if token.text not in stopwords and token.text not in punctuation]
tokens

['Natural',
 'Language',
 'Processing',
 'NLP',
 'sub',
 'field',
 'artificial',
 'intelligence',
 'AI',
 'focuses',
 'interaction',
 'computers',
 'humans',
 'natural',
 'language',
 'The',
 'ultimate',
 'objective',
 'NLP',
 'read',
 'decipher',
 'understand',
 'sense',
 'human',
 'language',
 'valuable',
 'way',
 'By',
 'utilizing',
 'NLP',
 'developers',
 'organize',
 'structure',
 'knowledge',
 'perform',
 'tasks',
 'automatic',
 'summarization',
 'translation',
 'named',
 'entity',
 'recognition',
 'relationship',
 'extraction',
 'sentiment',
 'analysis',
 'speech',
 'recognition',
 'topic',
 'segmentation']

# Getting word frequencies and normalizing


In [14]:
word_frequencies = {}
for word in doc:
    word = word.text.lower()
    if word in stopwords or word in punctuation:
        continue
    if word not in word_frequencies.keys():
        word_frequencies[word] = 1
    else:
        word_frequencies[word] += 1

max_frequency = max(word_frequencies.values())
# normalize the frequencies
for word in word_frequencies.keys():
    word_frequencies[word] = (word_frequencies[word] / max_frequency)

# Sentence Scores

In [15]:
sentence_scores = {}
sentence_tokens = [sent for sent in doc.sents]
for sent in doc.sents:
    for word in sent:
        if word.text.lower() in word_frequencies.keys():
            if sent not in sentence_scores:
                sentence_scores[sent] = word_frequencies[word.text.lower()]
            else:
                sentence_scores[sent] += word_frequencies[word.text.lower()]

print(sentence_scores)

{
Natural Language Processing (NLP) is a sub-field of artificial intelligence (AI) that focuses on the interaction between computers and humans through natural language.: 7.666666666666665, The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a valuable way.: 5.0, By utilizing NLP, developers can organize and structure knowledge to perform tasks such as automatic summarization, translation, named entity recognition, relationship extraction, sentiment analysis, speech recognition, and topic segmentation.
: 8.666666666666666}


# Summarization

In [16]:
sentence_scores = {k: v for k, v in sorted(sentence_scores.items(), key=lambda item: item[1], reverse=True)}

In [17]:
sentence_scores

{By utilizing NLP, developers can organize and structure knowledge to perform tasks such as automatic summarization, translation, named entity recognition, relationship extraction, sentiment analysis, speech recognition, and topic segmentation.: 8.666666666666666,
 
 Natural Language Processing (NLP) is a sub-field of artificial intelligence (AI) that focuses on the interaction between computers and humans through natural language.: 7.666666666666665,
 The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a valuable way.: 5.0}

In [18]:
select_length = int(len(sentence_tokens) * 0.3)
if select_length < 1:
    select_length = 1
summary_sentences = nlargest(select_length, sentence_scores, key=sentence_scores.get)
summary = ' '.join([sent.text for sent in summary_sentences])

In [19]:
print("Summary:", summary)

Summary: By utilizing NLP, developers can organize and structure knowledge to perform tasks such as automatic summarization, translation, named entity recognition, relationship extraction, sentiment analysis, speech recognition, and topic segmentation.



In [20]:
len_summary = len(summary.split())
print("Length of summary:", len_summary)
print("Length of original text:", len(text.split()))
print("Summary length in percentage:", (len_summary / len(text.split())) * 100)

Length of summary: 29
Length of original text: 72
Summary length in percentage: 40.27777777777778
